## Comparing Performance of Pareto-Fronts

Comparing performance of Pareto-Fronts constructed using:
- Reference Bitrate Ladder: Exhaustive Encoding
- Cross-Over Bitrates: From target Cross-Over bitrates used during training of models.
- Cross-Over Bitrates: From target Cross-Over VMAFs used during training of models.

against fixed bitrate ladder

### Imports

Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os, sys, warnings
import pickle
from tqdm import tqdm
warnings.filterwarnings("ignore")
sys.path.append("/home/kd28684/Constructing-Per-Shot-Bitrate-Ladders-using-Visual-Information-Fidelity-Working")
import functions.IO_functions as IO_functions
import functions.extract_functions as extract_functions
import modules.bitrate_ladder_prediction_dataset_functions as bitrate_ladder_prediction_dataset_functions
import modules.quality_ladder_prediction_dataset_functions as quality_ladder_prediction_dataset_functions
import modules.bitrate_quality_ladder_evaluation_functions as bitrate_quality_ladder_evaluation_functions
import defaults

### Performance of Reference Bitrate Ladders against the Fixed Bitrate Ladder

In [ ]:
Metrics = np.load("../results/main/Standard/bd_metrics/Reference_Bitrate_Ladder/libx265_medium.npy", allow_pickle=True)[()]
Metrics = np.asarray(list(Metrics.values()))
Metrics = Metrics[np.logical_not(np.all(np.isnan(Metrics), axis=1)), :]
mean = np.round(np.mean(Metrics, axis=0), decimals=3)
std = np.round(np.std(Metrics, axis=0), decimals=3)

# Histogram plot of BD-metrics
plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
plt.title(r"BD-Rate wrt AL ($\mu$={}, $\sigma$={})".format(mean[0], std[0]))
plt.grid()
sns.histplot(data=Metrics[:,0], bins=20, kde=True, element="step")

plt.subplot(2,2,2)
plt.title(r"BD-VMAF wrt AL ($\mu$={}, $\sigma$={})".format(mean[1], std[1]))
plt.grid()
sns.histplot(data=Metrics[:,1], bins=20, kde=True, element="step")

plt.subplot(2,2,3)
plt.title(r"BD-Rate wrt RL ($\mu$={}, $\sigma$={})".format(mean[2], std[2])) 
plt.grid()
sns.histplot(data=Metrics[:,2], bins=20, kde=True, element="step")

plt.subplot(2,2,4)
plt.title(r"BD-VMAF wrt RL ($\mu$={}, $\sigma$={})".format(mean[3], std[3]))
plt.grid()
sns.histplot(data=Metrics[:,3], bins=20, kde=True, element="step")

plt.show()

### Constructing True Cross-Over Bitrate Ladder

In [3]:
# Parameters
codec = "libx265"
preset = "medium"
quality_metric = "vmaf"
evaluation_bitrates = defaults.evaluation_bitrates
Resolutions = defaults.resolutions

# Features (We don't need to extract the right features)
features_names = []
for features_subset in [defaults.si_features]:
	for f in features_subset:
		features_names.append(f)

Video_Bitrate_Ladders = {}
for video_file in defaults.Test_Video_Titles:
	# Get True Cross-Over Bitrates
	CrossOver_Bitrates = []
	for i in range(len(Resolutions)-1):	
		_, cp = bitrate_ladder_prediction_dataset_functions.LowLevelFeatures_CrossOverBitrates_Dataset(
				codec=codec,
				preset=preset,
				quality_metric=quality_metric,
				features_names=features_names,
				video_filenames=[video_file],
				temporal_low_level_features=False,
				Resolutions_Considered=Resolutions,
				CRFs_Considered=defaults.CRFs,
				QPs_Considered=None,
				high_res=Resolutions[i],
				low_res=Resolutions[i+1],
				min_quality=defaults.min_quality,
				max_quality=defaults.max_quality,
				min_bitrate=defaults.min_bitrate,
				max_bitrate=defaults.max_bitrate
			)
		CrossOver_Bitrates.append(cp)
	
	# Calculating Bitrate-Ladder
	Bitrate_Ladder = {}
	for i in range(len(evaluation_bitrates)):
		# Switching happens to higher resolution when bitrate >= crossover_bitrate of corresponding higher resolution.
		b = evaluation_bitrates[i]
		Bitrate_Ladder[b] = None

		for j in range(1+len(CrossOver_Bitrates)):
			if (j==0) and (b >= CrossOver_Bitrates[j]):
				Bitrate_Ladder[b] = Resolutions[0]
			elif (j <= len(CrossOver_Bitrates)-1) and (CrossOver_Bitrates[j] <= b < CrossOver_Bitrates[j-1]):
				Bitrate_Ladder[b] = Resolutions[j]
			elif (j==len(CrossOver_Bitrates)) and (b < CrossOver_Bitrates[j-1]):
				Bitrate_Ladder[b] = Resolutions[-1]
			else:
				None

		if Bitrate_Ladder[b] is None:
			assert False, "Something is Wrong"

	Video_Bitrate_Ladders[video_file] = Bitrate_Ladder

np.save("logs/True_CrossOver_Bitrate_Ladders.npy", Video_Bitrate_Ladders)

In [ ]:
Metrics = []
for video_file in defaults.Test_Video_Titles:
	Metrics.append(bitrate_quality_ladder_evaluation_functions.Calculate_BD_metrics_for_Bitrate_Ladders(
		video_file=video_file,
		codec=codec,
		preset=preset,
		bitrate_ladder_path="logs/True_CrossOver_Bitrate_Ladders.npy",
		fixed_bitrate_ladder_path="../results/main/Standard/bitrate_ladders/fixed_bitrate_ladder.npy",
		reference_bitrate_ladder_path="../results/main/Standard/bitrate_ladders/libx265_medium.npy"
	))
	
Metrics = np.asarray(Metrics)
Metrics = Metrics[np.logical_not(np.all(np.isnan(Metrics), axis=1)), :]
mean = np.round(np.mean(Metrics, axis=0), decimals=3)
std = np.round(np.std(Metrics, axis=0), decimals=3)

# Histogram plot of BD-metrics
plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
plt.title(r"BD-Rate wrt AL ($\mu$={}, $\sigma$={})".format(mean[0], std[0]))
plt.grid()
sns.histplot(data=Metrics[:,0], bins=20, kde=True, element="step")

plt.subplot(2,2,2)
plt.title(r"BD-VMAF wrt AL ($\mu$={}, $\sigma$={})".format(mean[1], std[1]))
plt.grid()
sns.histplot(data=Metrics[:,1], bins=20, kde=True, element="step")

plt.subplot(2,2,3)
plt.title(r"BD-Rate wrt RL ($\mu$={}, $\sigma$={})".format(mean[2], std[2]))
plt.grid()
sns.histplot(data=Metrics[:,2], bins=20, kde=True, element="step")

plt.subplot(2,2,4)
plt.title(r"BD-VMAF wrt RL ($\mu$={}, $\sigma$={})".format(mean[3], std[3]))
plt.grid()
sns.histplot(data=Metrics[:,3], bins=20, kde=True, element="step")

plt.show()

### Constructing True Cross-Over Quality Ladder

In [5]:
# Parameters
codec = "libx265"
preset = "medium"
quality_metric = "vmaf"
evaluation_qualities = defaults.evaluation_qualities
Resolutions = defaults.resolutions

# Features (We don't need to extract the right features)
features_names = []
for features_subset in [defaults.si_features]:
	for f in features_subset:
		features_names.append(f)

Video_Quality_Ladders = {}
for video_file in defaults.Test_Video_Titles:
	# Get True Cross-Over Bitrates
	CrossOver_Bitrates = []
	for i in range(len(Resolutions)-1):	
		_, cp = quality_ladder_prediction_dataset_functions.LowLevelFeatures_CrossOverQualities_Dataset(
				codec=codec,
				preset=preset,
				quality_metric=quality_metric,
				features_names=features_names,
				video_filenames=[video_file],
				temporal_low_level_features=False,
				Resolutions_Considered=Resolutions,
				CRFs_Considered=defaults.CRFs,
				QPs_Considered=None,
				high_res=Resolutions[i],
				low_res=Resolutions[i+1],
				min_quality=defaults.min_quality,
				max_quality=defaults.max_quality,
				min_bitrate=defaults.min_bitrate,
				max_bitrate=defaults.max_bitrate
			)
		CrossOver_Bitrates.append(cp)
	
	# Calculating Bitrate-Ladder
	Quality_Ladder = {}
	for i in range(len(evaluation_qualities)):
		# Switching happens to higher resolution when bitrate >= crossover_bitrate of corresponding higher resolution.
		b = evaluation_qualities[i]
		Quality_Ladder[b] = None

		for j in range(1+len(CrossOver_Bitrates)):
			if (j==0) and (b >= CrossOver_Bitrates[j]):
				Quality_Ladder[b] = Resolutions[0]
			elif (j <= len(CrossOver_Bitrates)-1) and (CrossOver_Bitrates[j] <= b < CrossOver_Bitrates[j-1]):
				Quality_Ladder[b] = Resolutions[j]
			elif (j==len(CrossOver_Bitrates)) and (b < CrossOver_Bitrates[j-1]):
				Quality_Ladder[b] = Resolutions[-1]
			else:
				None

		if Quality_Ladder[b] is None:
			assert False, "Something is Wrong"

	Video_Quality_Ladders[video_file] = Quality_Ladder

np.save("logs/True_CrossOver_Quality_Ladders.npy", Video_Quality_Ladders)

In [ ]:
Metrics = []
for video_file in defaults.Test_Video_Titles:
	Metrics.append(bitrate_quality_ladder_evaluation_functions.Calculate_BD_metrics_for_Quality_Ladders(
		video_file=video_file,
		codec=codec,
		preset=preset,
		quality_ladder_path="logs/True_CrossOver_Quality_Ladders.npy",
		fixed_bitrate_ladder_path="../results/main/Standard/bitrate_ladders/fixed_bitrate_ladder.npy",
		reference_bitrate_ladder_path="../results/main/Standard/bitrate_ladders/libx265_medium.npy"
	))
	
Metrics = np.asarray(Metrics)
Metrics = Metrics[np.logical_not(np.all(np.isnan(Metrics), axis=1)), :]
mean = np.round(np.mean(Metrics, axis=0), decimals=3)
std = np.round(np.std(Metrics, axis=0), decimals=3)

# Histogram plot of BD-metrics
plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
plt.title(r"BD-Rate wrt AL ($\mu$={}, $\sigma$={})".format(mean[0], std[0]))
plt.grid()
sns.histplot(data=Metrics[:,0], bins=20, kde=True, element="step")

plt.subplot(2,2,2)
plt.title(r"BD-VMAF wrt AL ($\mu$={}, $\sigma$={})".format(mean[1], std[1]))
plt.grid()
sns.histplot(data=Metrics[:,1], bins=20, kde=True, element="step")

plt.subplot(2,2,3)
plt.title(r"BD-Rate wrt RL ($\mu$={}, $\sigma$={})".format(mean[2], std[2]))
plt.grid()
sns.histplot(data=Metrics[:,2], bins=20, kde=True, element="step")

plt.subplot(2,2,4)
plt.title(r"BD-VMAF wrt RL ($\mu$={}, $\sigma$={})".format(mean[3], std[3]))
plt.grid()
sns.histplot(data=Metrics[:,3], bins=20, kde=True, element="step")

plt.show()